# 27i Teacher Archaeology

Mine older DQA-MoX checkpoints from 24a/25a/27a on the official scene-daynight total split. Goal: find a stronger teacher anchor before designing the next MoE training loop.

In [ ]:
from pathlib import Path
import csv
import shutil
import subprocess
import sys
from datetime import datetime, timezone

REPO_ROOT = Path('/app/Object_Detection')
SCENE_ROOT = REPO_ROOT / 'dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa'
AGG_ROOT = SCENE_ROOT / 'aggressive_dqamox'
WORKSPACE = AGG_ROOT / 'output/27i_teacher_archaeology'
EVAL = SCENE_ROOT / 'scripts/evaluate_scene_daynight_protocol.py'
REPORTS = AGG_ROOT / 'reports'

def ck(label, rel):
    path = REPO_ROOT / rel
    if not path.exists():
        print('missing:', label, path)
        return None
    return f'{label}={path}'

candidates = [
    ck('warmup_08', 'dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/08_full_latent_dqamox_from_warmup/checkpoints/round000_latent_dqamox_warmup.pt'),
    ck('24a_p2r3_aggregate', 'dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/24_aggressive_until_target/24a_client_dominant_soft_expand/checkpoints/latent_dqamox_p2_round003_dqa_aggregate.pt'),
    ck('24a_p2r3_repair', 'dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/24_aggressive_until_target/24a_client_dominant_soft_expand/checkpoints/latent_dqamox_p2_round003_server_repair.pt'),
    ck('25a_r1_aggregate', 'dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/25_paper_round_until_target/25a_fedmox50_sto20_30_top1/checkpoints/latent_dqamox_p1_round001_dqa_aggregate.pt'),
    ck('25a_r1_repair', 'dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/25_paper_round_until_target/25a_fedmox50_sto20_30_top1/checkpoints/latent_dqamox_p1_round001_server_repair.pt'),
    ck('25a_r5_aggregate', 'dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/25_paper_round_until_target/25a_fedmox50_sto20_30_top1/checkpoints/latent_dqamox_p1_round005_dqa_aggregate.pt'),
    ck('25a_r5_repair', 'dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/25_paper_round_until_target/25a_fedmox50_sto20_30_top1/checkpoints/latent_dqamox_p1_round005_server_repair.pt'),
    ck('25a_r10_aggregate', 'dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/25_paper_round_until_target/25a_fedmox50_sto20_30_top1/checkpoints/latent_dqamox_p1_round010_dqa_aggregate.pt'),
    ck('25a_r10_repair', 'dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/25_paper_round_until_target/25a_fedmox50_sto20_30_top1/checkpoints/latent_dqamox_p1_round010_server_repair.pt'),
    ck('25a_r15_aggregate', 'dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/25_paper_round_until_target/25a_fedmox50_sto20_30_top1/checkpoints/latent_dqamox_p1_round015_dqa_aggregate.pt'),
    ck('27a_r5_aggregate', 'dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27a_soft_mixture_head_first_40_10/checkpoints/latent_dqamox_p1_round005_dqa_aggregate.pt'),
    ck('27a_r5_repair', 'dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27a_soft_mixture_head_first_40_10/checkpoints/latent_dqamox_p1_round005_server_repair.pt'),
    ck('27a_r10_aggregate', 'dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27a_soft_mixture_head_first_40_10/checkpoints/latent_dqamox_p1_round010_dqa_aggregate.pt'),
    ck('27a_r10_repair', 'dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27a_soft_mixture_head_first_40_10/checkpoints/latent_dqamox_p1_round010_server_repair.pt'),
    ck('27a_r13_aggregate', 'dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27a_soft_mixture_head_first_40_10/checkpoints/latent_dqamox_p1_round013_dqa_aggregate.pt'),
    ck('27a_r13_repair', 'dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27a_soft_mixture_head_first_40_10/checkpoints/latent_dqamox_p1_round013_server_repair.pt'),
]
candidates = [c for c in candidates if c]

WORKSPACE.mkdir(parents=True, exist_ok=True)
cmd = [
    sys.executable, str(EVAL),
    '--workspace', str(WORKSPACE),
    '--splits', 'total',
    '--batch-size', '16',
    '--no-plots',
    '--verbose',
]
for spec in candidates:
    cmd.extend(['--checkpoint', spec])
print('Evaluating', len(candidates), 'checkpoints')
proc = subprocess.run(cmd, cwd=REPO_ROOT, text=True)
print('Return code:', proc.returncode)

summary_csv = WORKSPACE / 'validation_reports/paper_protocol_eval_summary.csv'
rows = []
with summary_csv.open(newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    rows = [r for r in reader if r.get('status') == 'ok']
rows.sort(key=lambda r: (float(r['map50']), float(r['map50_95'])), reverse=True)
stats_dir = WORKSPACE / 'stats'
stats_dir.mkdir(parents=True, exist_ok=True)
out_csv = stats_dir / '27i_teacher_archaeology_total_metrics.csv'
shutil.copy2(summary_csv, out_csv)
best = rows[0]
print('Best:', best['checkpoint_label'], best['map50'], best['map50_95'])
for r in rows[:8]:
    print(r['checkpoint_label'], r['map50'], r['map50_95'])

REPORTS.mkdir(parents=True, exist_ok=True)
research_csv = REPORTS / '27_research_loop_mAP_summary.csv'
fieldnames = ['trial','status','best_map50','best_map50_95','warmup_map50','repair_map50','dqa_aggregate_map50','dqa_repair_map50','workspace','notebook','log','finished_utc','rationale']
exists = research_csv.exists()
with research_csv.open('a', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    if not exists:
        writer.writeheader()
    writer.writerow({
        'trial': '27i_teacher_archaeology',
        'status': 'target_reached' if float(best['map50']) >= 0.60 else 'completed',
        'best_map50': best['map50'],
        'best_map50_95': best['map50_95'],
        'warmup_map50': next((r['map50'] for r in rows if r['checkpoint_label'] == 'warmup_08'), ''),
        'workspace': str(WORKSPACE),
        'notebook': str(AGG_ROOT / 'notebooks/research_loop_until_060/003_27i_teacher_archaeology.ipynb'),
        'log': best.get('log_file', ''),
        'finished_utc': datetime.now(timezone.utc).isoformat(),
        'rationale': 'Mine long 24a/25a/27a checkpoints before training another MoE; if an older teacher is stronger, use it as the next anchor.',
    })

try:
    sys.path.insert(0, str(REPO_ROOT))
    from notebook_notify import notify_discord
    msg = '\n'.join([
        '27i teacher archaeology finished.',
        f"Best: {best['checkpoint_label']} mAP50={best['map50']} / mAP50:95={best['map50_95']}",
        f'CSV: {out_csv}',
        'Decision: ' + ('target reached' if float(best['map50']) >= 0.60 else 'no old checkpoint breaks the ceiling; train a new stronger teacher next'),
    ])
    print(notify_discord(msg, title='DQA-MoX 27i result', fail_silently=True))
except Exception as exc:
    print('Discord skipped:', exc)

if proc.returncode != 0:
    raise SystemExit(proc.returncode)
